Erklärung deines Dash-Dashboard-Codes
1. Imports und Daten laden
python
Kopieren
Bearbeiten
import dash
from dash import Dash, dcc, html, Input, Output, State
import pandas as pd
import os
import sys
import plotly.express as px
dash & Dash: Dash ist das Framework für Web-Dashboards in Python.

dcc & html: Dash-Komponenten (Interaktive Komponenten wie Dropdowns, Slider und HTML-Elemente).

Input, Output, State: Für die Callback-Logik, die die App interaktiv macht.

pandas: Für Datenhandling.

os, sys: Für Dateipfade und das Einbinden eigener Module.

plotly.express: Für einfache Erstellung von Diagrammen.

2. Daten einlesen
python
Kopieren
Bearbeiten
csv_ordner = r"C:\DataCraft\11_Datenvisualisierung\Projekt-Emission-Dashboard\data\raw"
data_sources = {}
for dateiname in os.listdir(csv_ordner):
    if dateiname.endswith(".csv"):
        dateipfad = os.path.join(csv_ordner, dateiname)
        df = pd.read_csv(dateipfad)
        key = os.path.splitext(dateiname)[0]
        data_sources[key] = df
Hier werden alle .csv-Dateien in deinem Datenordner eingelesen.

Jede CSV wird in ein pandas DataFrame geladen und in data_sources als Dictionary gespeichert, der Schlüssel ist der Dateiname ohne .csv.

So hast du Zugriff auf alle Datensätze dynamisch.

3. Import des Länder-Kontinent-Mappings
python
Kopieren
Bearbeiten
sys.path.append(r"C:\DataCraft\11_Datenvisualisierung\Projekt-Emission-Dashboard\data")
from kontinente_laender_liste import country_to_continent_map
Dieses Mapping ist ein Python-Dictionary, das jedem Land einen Kontinent zuordnet.

So kannst du Länder nach Kontinenten filtern und Gruppierungen im Dashboard machen.

4. Initialisierung der Dash-App
python
Kopieren
Bearbeiten
app = Dash(__name__)
Grundgerüst, mit dem Dash weiß, dass hier eine App gestartet wird.

5. Layout der App
a) Sidebar (aufklappbar) für Auswahl der Emissionsdaten
Die Sidebar ist ein html.Div mit CSS für Positionierung und Animations-„Sliding“ von links.

Buttons zum Öffnen/Schließen der Sidebar.

Auswahl-Dropdown für die Emissionsquelle (Wahl aus den CSV-Dateien).

b) Hauptbereich
Oben Dropdowns für Kontinent- und Länder-Auswahl mit Hinweisen als Text.

Darunter ein RangeSlider zur Auswahl des Jahresbereichs.

Dann 4 Graphen in einem 2x2 Grid (line, bar, pie, scatter).

Das Layout sorgt für responsive und optisch getrennte Bereiche.

6. Callbacks (das Herzstück)
a) Sidebar toggeln
Öffnet oder schließt die Sidebar, ohne dass der Hauptbereich verschoben wird.

Nutzt dash.callback_context, um zu wissen, welcher Button geklickt wurde.

Passt CSS-Eigenschaften (left) an, um die Sidebar rein/raus zu schieben.

b) Emissions-Datenquelle wechseln
Wenn du in der Sidebar eine andere CSV auswählst, wird diese in der App verwendet.

Aktualisiert automatisch alle weiteren Dropdowns und Graphen.

c) Kontinent zu Länder Dropdown synchronisieren
Je nachdem, welcher Kontinent gewählt ist, werden nur Länder dieses Kontinents im Länder-Dropdown angezeigt.

So kannst du gezielt Länder auswählen.

d) Graphen aktualisieren
Basierend auf den gewählten Ländern, Jahresbereich und Emissionsdaten.

Filtert den DataFrame entsprechend.

Erstellt mit Plotly Express vier unterschiedliche Visualisierungen:

Liniendiagramm (Verlauf über die Jahre)

Balkendiagramm (für Vergleich)

Kreisdiagramm (Pie, Verteilung im ersten ausgewählten Jahr)

Streudiagramm (scatter)

7. Warum so?
Modularität: Die Aufteilung in Sidebar, Dropdowns und Graphen macht das Dashboard übersichtlich.

Interaktivität: Callbacks sorgen dafür, dass Benutzer Eingaben machen können und sofort Ergebnisse sehen.

Dynamik: Neue Datenquellen lassen sich einfach durch neue CSVs hinzufügen.

UX: Sidebar klappt aus ohne den Hauptinhalt zu verschieben, Dropdowns haben Hinweise für bessere Verständlichkeit.

Saubere Struktur: Mit CSS Grid für die Graphen wird Platz effizient genutzt.

8. Tipps zum Selbstlernen
Schau dir einzelne Callbacks an und teste, was passiert, wenn du Eingaben änderst.

Probiere aus, wie Filter auf DataFrames funktionieren.

Experimentiere mit Plotly Express: Ändere Diagrammtypen oder Achsen.

Modifiziere CSS-Eigenschaften für das Aussehen.

Erweitere das Dashboard um neue Funktionen (z.B. mehr Filter, andere Graphen).



Klar, hier bekommst du den kompletten finalen Code mit ausführlichen Kommentaren und Erklärungen direkt im Code – so kannst du alles Schritt für Schritt nachvollziehen und direkt lernen:

python
Kopieren
Bearbeiten
# Import der notwendigen Bibliotheken
import dash
from dash import Dash, dcc, html, Input, Output, State
import pandas as pd
import os
import sys
import plotly.express as px

# -----------------------------
# 1. Daten laden
# -----------------------------

# Ordner mit den CSV-Dateien (ersetze durch deinen Pfad)
csv_ordner = r"C:\DataCraft\11_Datenvisualisierung\Projekt-Emission-Dashboard\data\raw"

# Dictionary zum Speichern der geladenen DataFrames, Schlüssel = Dateiname ohne .csv
data_sources = {}
for dateiname in os.listdir(csv_ordner):
    if dateiname.endswith(".csv"):
        dateipfad = os.path.join(csv_ordner, dateiname)
        df = pd.read_csv(dateipfad)
        key = os.path.splitext(dateiname)[0]
        data_sources[key] = df

# -----------------------------
# 2. Länder-Kontinent Mapping laden
# -----------------------------

# Pfad anpassen: Datei mit Mapping dict (country_to_continent_map)
sys.path.append(r"C:\DataCraft\11_Datenvisualisierung\Projekt-Emission-Dashboard\data")
from kontinente_laender_liste import country_to_continent_map

# Alle einzigartigen Kontinente für Dropdown
kontinente = sorted(set(country_to_continent_map.values()))

# -----------------------------
# 3. Dash App initialisieren
# -----------------------------

app = Dash(__name__)

# -----------------------------
# 4. Hilfsfunktionen
# -----------------------------

# Länder nach Kontinent filtern
def get_countries_by_continent(continent):
    return sorted([country for country, cont in country_to_continent_map.items() if cont == continent])

# Liste aller Emissionsdatensätze (Schlüssel aus data_sources) für Dropdown
emission_options = [{'label': k.replace('-', ' ').title(), 'value': k} for k in sorted(data_sources.keys())]

# -----------------------------
# 5. App Layout
# -----------------------------

app.layout = html.Div([
    # Sidebar - ausklappbar von links
    html.Div([
        html.Button("☰ Emissionsquelle wählen", id="open-sidebar", n_clicks=0,
                    style={'fontSize': '18px', 'marginBottom': '10px'}),
        html.Div([
            html.H4("Emissionsdaten auswählen"),
            dcc.Dropdown(
                id='emission-dropdown',
                options=emission_options,
                value=emission_options[0]['value'] if emission_options else None,
                clearable=False,
                style={'fontSize': '16px'}
            ),
            html.Button("Schließen ✖", id="close-sidebar", n_clicks=0,
                        style={'marginTop': '15px'})
        ], id='sidebar-content',
           style={
               'padding': '15px',
               'fontSize': '16px'
           })
    ], id='sidebar',
       style={
           'position': 'fixed',
           'top': 0,
           'left': '-250px',  # start versteckt links außerhalb
           'width': '250px',
           'height': '100%',
           'backgroundColor': '#f8f9fa',
           'borderRight': '1px solid #ddd',
           'boxShadow': '2px 0 5px rgba(0,0,0,0.1)',
           'transition': 'left 0.3s ease',
           'zIndex': 1000
       }),

    # Hauptbereich - nimmt ganze Breite ein, ohne von Sidebar verschoben zu werden
    html.Div([
        # Obere Filter (Kontinent, Länder) mit Hinweisen
        html.Div([
            html.Div("Es können bis zu alle Kontinente ausgewählt werden:", style={'marginBottom': '5px', 'fontWeight': 'bold'}),
            dcc.Dropdown(
                id='kontinent-dropdown',
                options=[{'label': k, 'value': k} for k in kontinente],
                value=kontinente[0],
                clearable=False,
                multi=True,
                style={'marginBottom': '15px'}
            ),
            html.Div("Es können maximal fünf Länder ausgewählt werden:", style={'marginBottom': '5px', 'fontWeight': 'bold'}),
            dcc.Dropdown(
                id='country-dropdown',
                options=[],  # wird dynamisch gefüllt
                value=[],
                multi=True,
                style={'marginBottom': '15px'}
            )
        ], style={'width': '60%', 'margin': 'auto'}),

        # Jahresbereich Slider (RangeSlider)
        html.Div(
            dcc.RangeSlider(
                id='year-slider',
                min=1990,  # Platzhalter - wird später aktualisiert
                max=2020,
                value=[1990, 2020],
                marks={},  # wird dynamisch gesetzt
                step=1,
                allowCross=False,
                tooltip={"placement": "bottom", "always_visible": True},
            ),
            style={'width': '60%', 'margin': '40px auto 40px auto'}
        ),

        # 4 Graphen im 2x2 Grid
        html.Div([
            dcc.Graph(id='graph-1'),
            dcc.Graph(id='graph-2'),
            dcc.Graph(id='graph-3'),
            dcc.Graph(id='graph-4'),
        ], style={
            'display': 'grid',
            'gridTemplateColumns': '50% 50%',
            'gridGap': '20px',
            'width': '90%',
            'margin': 'auto',
        })
    ], style={'marginLeft': '0', 'transition': 'margin-left 0.3s ease'}),
])

# -----------------------------
# 6. Callbacks
# -----------------------------

# Sidebar öffnen/schließen ohne Inhalt zu verschieben
@app.callback(
    Output('sidebar', 'style'),
    Input('open-sidebar', 'n_clicks'),
    Input('close-sidebar', 'n_clicks'),
    State('sidebar', 'style'),
)
def toggle_sidebar(open_clicks, close_clicks, style):
    ctx = dash.callback_context
    if not ctx.triggered:
        return style
    button_id = ctx.triggered[0]['prop_id'].split('.')[0]
    if button_id == 'open-sidebar':
        style['left'] = '0'
    elif button_id == 'close-sidebar':
        style['left'] = '-250px'
    return style

# Wenn eine neue Emissionsquelle gewählt wird:
# - Update der Jahresspanne und Marks des Sliders
# - Aktualisierung der Dropdowns
@app.callback(
    Output('year-slider', 'min'),
    Output('year-slider', 'max'),
    Output('year-slider', 'value'),
    Output('year-slider', 'marks'),
    Output('kontinent-dropdown', 'options'),
    Output('kontinent-dropdown', 'value'),
    Input('emission-dropdown', 'value'),
)
def update_years_and_kontinente(emission_key):
    if emission_key is None:
        return dash.no_update, dash.no_update, dash.no_update, dash.no_update, dash.no_update, dash.no_update

    df = data_sources.get(emission_key)
    if df is None or df.empty:
        return dash.no_update, dash.no_update, dash.no_update, dash.no_update, dash.no_update, dash.no_update

    min_year = int(df['Year'].min())
    max_year = int(df['Year'].max())

    # Erstelle Marks für Slider (alle 5 Jahre)
    marks = {str(y): str(y) for y in range(min_year, max_year + 1, 5)}

    # Kontinent Optionen bleiben gleich, setze default alle ausgewählt
    kontinent_opts = [{'label': k, 'value': k} for k in kontinente]

    return min_year, max_year, [min_year, max_year], marks, kontinent_opts, kontinente

# Aktualisiere Länder-Dropdown basierend auf gewählten Kontinenten
@app.callback(
    Output('country-dropdown', 'options'),
    Output('country-dropdown', 'value'),
    Input('kontinent-dropdown', 'value'),
    State('country-dropdown', 'value')
)
def update_countries(selected_kontinente, selected_länder):
    if not selected_kontinente:
        return [], []

    if isinstance(selected_kontinente, str):
        selected_kontinente = [selected_kontinente]

    # Alle Länder aus allen ausgewählten Kontinenten sammeln
    länder = []
    for kontinent in selected_kontinente:
        länder += get_countries_by_continent(kontinent)
    länder = sorted(set(länder))

    options = [{'label': land, 'value': land} for land in länder]

    # Länder-Auswahl auf max. 5 begrenzen, falls mehr gewählt waren, kürzen
    if selected_länder:
        new_value = [land for land in selected_länder if land in länder][:5]
    else:
        # Standardwert: die ersten 5 Länder
        new_value = länder[:5]

    return options, new_value

# Graphen updaten basierend auf Auswahl von Ländern, Jahr und Emissionsdaten
@app.callback(
    Output('graph-1', 'figure'),
    Output('graph-2', 'figure'),
    Output('graph-3', 'figure'),
    Output('graph-4', 'figure'),
    Input('country-dropdown', 'value'),
    Input('year-slider', '